# Fetal Health Risk Classification
## XGBoost · SMOTE · SHAP Explainability

**Author:** Arpi | Applied Mathematics, University of Dhaka  
**Domain:** Child health · Clinical machine learning  

---

This notebook builds an end-to-end machine learning pipeline to classify fetal health status
(Normal / Suspect / Pathological) from cardiotocography (CTG) measurements. It addresses
class imbalance via SMOTE, compares multiple classifiers, evaluates with ROC-AUC and F1
metrics, and explains predictions using SHAP values — meeting clinical interpretability requirements.

**Contents:**
1. Setup & Imports
2. Data Loading & Exploration
3. Class Imbalance Analysis
4. Train/Test Split & SMOTE
5. Model Training (Logistic Regression, Random Forest, XGBoost)
6. Evaluation: Confusion Matrix
7. Evaluation: ROC Curves
8. Feature Importance
9. SHAP Explainability
10. Model Comparison
11. Model Persistence
12. Clinical Summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

TEAL='#0F6E56'; CORAL='#D85A30'; BLUE='#185FA5'; AMBER='#BA7517'; GRAY='#5F5E5A'
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.facecolor': 'white', 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'font.size': 11, 'axes.titlesize': 13
})
print('Environment ready.')

## 2. Data Loading & Exploration

In [ ]:
df = pd.read_csv('../data/fetal_health_bangladesh.csv')
print(f'Shape: {df.shape}')
print(f'\nClass distribution:')
class_map = {1: 'Normal', 2: 'Suspect', 3: 'Pathological'}
print(df['fetal_health'].map(class_map).value_counts())
print(f'\nClass distribution (%):')
print((df['fetal_health'].map(class_map).value_counts(normalize=True)*100).round(1))
df.head()

In [ ]:
print('=== Descriptive Statistics ===')
df.describe().round(3)

In [ ]:
# Correlation heatmap (top features)
fig, ax = plt.subplots(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.3, linecolor='white', ax=ax, annot=False,
            cbar_kws={'label': 'Correlation', 'shrink': 0.8})
ax.set_title('Feature Correlation Matrix — CTG Features + Fetal Health', fontweight='bold', pad=12)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()

## 3. Class Imbalance Analysis

In [ ]:
class_counts = df['fetal_health'].value_counts().sort_index()
labels = ['Normal (1)', 'Suspect (2)', 'Pathological (3)']
colors = [TEAL, AMBER, CORAL]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(labels, class_counts.values, color=colors, edgecolor='white')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontsize=10, color=GRAY)
axes[0].set_title('Class Distribution — Raw Dataset', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie(class_counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1})
axes[1].set_title('Class Distribution (Proportional)', fontweight='bold')
plt.suptitle('Fetal Health Class Imbalance — Before Correction', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()
print('\nClass imbalance ratio (majority:minority) =', f'{class_counts.max() / class_counts.min():.1f}:1')

## 4. Train/Test Split & SMOTE Balancing

**Critical:** SMOTE is applied **only to the training set** to prevent data leakage.
The test set reflects the true real-world class distribution.

In [ ]:
X = df.drop('fetal_health', axis=1)
y = df['fetal_health'] - 1  # Encode: 0=Normal, 1=Suspect, 2=Pathological

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} records')
print(f'Test set: {X_test.shape[0]} records')
print(f'Test class distribution: {pd.Series(y_test).value_counts().to_dict()}')

# Apply SMOTE only to training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'\nAfter SMOTE — training set: {X_train_res.shape[0]} records')
print(f'Balanced class counts: {pd.Series(y_train_res).value_counts().to_dict()}')

In [ ]:
labels_plot = ['Normal', 'Suspect', 'Pathological']
orig_counts = pd.Series(y_train).value_counts().sort_index().values
smt_counts  = pd.Series(y_train_res).value_counts().sort_index().values

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in zip(axes, [orig_counts, smt_counts],
                              ['Before SMOTE (Training set)', 'After SMOTE (Balanced)']):
    bars = ax.bar(labels_plot, counts, color=[TEAL, AMBER, CORAL], edgecolor='white')
    for bar, val in zip(bars, counts):
        ax.text(bar.get_x()+bar.get_width()/2, val+5, str(val), ha='center', fontsize=10)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
plt.suptitle('SMOTE Class Balancing', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/fig5_smote_balancing.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Training

Three models are trained and compared:
- **Logistic Regression** — linear baseline
- **Random Forest** — ensemble baseline
- **XGBoost** — gradient boosting (primary model)

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_res, y_train_res)
print('Logistic Regression trained.')

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
print('Random Forest trained.')

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_res, y_train_res)
print('XGBoost trained.')

## 6. Evaluation: Confusion Matrix

In [ ]:
y_pred = xgb_model.predict(X_test)
class_names = ['Normal', 'Suspect', 'Pathological']

print('=== Classification Report — XGBoost ===')
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='white', ax=ax, annot_kws={'size': 11})
for i in range(3):
    for j in range(3):
        ax.text(j+0.5, i+0.72, f'(n={cm[i,j]})', ha='center', fontsize=8, color='gray')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('XGBoost Confusion Matrix\n(Normalized; raw counts in parentheses)', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../outputs/figures/fig1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation: ROC Curves (One-vs-Rest)

In [ ]:
y_prob = xgb_model.predict_proba(X_test)
y_bin  = label_binarize(y_test, classes=[0, 1, 2])
colors_roc = [TEAL, AMBER, CORAL]

fig, ax = plt.subplots(figsize=(8, 6))
for i, (lbl, col) in enumerate(zip(class_names, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    auc = roc_auc_score(y_bin[:, i], y_prob[:, i])
    ax.plot(fpr, tpr, color=col, linewidth=2, label=f'{lbl} (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, alpha=0.6, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — One-vs-Rest\nXGBoost Fetal Health Classifier', fontweight='bold', pad=12)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/figures/fig2_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Importance (XGBoost Gain)

In [ ]:
imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=True)
top15 = imp.tail(15)

bar_colors = [CORAL if v > imp.quantile(0.85) else BLUE if v > imp.quantile(0.65) else TEAL
              for v in top15.values]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top15.index, top15.values, color=bar_colors, edgecolor='white', height=0.65)
ax.set_xlabel('Feature Importance (XGBoost gain)')
ax.set_title('Top 15 Feature Importances — Fetal Health Prediction\nXGBoost Classifier', fontweight='bold', pad=12)
patches = [
    mpatches.Patch(color=CORAL, label='High importance (>85th pct)'),
    mpatches.Patch(color=BLUE,  label='Moderate (65–85th pct)'),
    mpatches.Patch(color=TEAL,  label='Standard')
]
ax.legend(handles=patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/fig3_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. SHAP Explainability

SHAP (SHapley Additive exPlanations) quantifies each feature's contribution to individual
predictions. TreeExplainer provides exact Shapley values for XGBoost without sampling.

This is a critical step for clinical deployment — clinicians must understand *why* a model
flags a CTG trace as pathological.

In [ ]:
explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test[:200])

# Summary bar plot (per class)
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_vals, X_test[:200], plot_type='bar',
                  class_names=['Normal', 'Suspect', 'Pathological'],
                  show=False, max_display=12)
plt.title('SHAP Feature Impact — Mean |SHAP value| by Class', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../outputs/figures/fig4_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm plot for Pathological class (index 2)
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_vals[2], X_test[:200], show=False, max_display=12)
plt.title('SHAP Beeswarm — Pathological Class\n(Feature impact on P(Pathological))',
          fontweight='bold', pad=12)
plt.tight_layout()
plt.show()
print('\nTop 5 features for Pathological classification (by mean |SHAP|):')
top_shap = pd.Series(
    np.abs(shap_vals[2]).mean(axis=0), index=X.columns
).sort_values(ascending=False).head(5)
print(top_shap.round(4))

## 10. Model Comparison

In [ ]:
# Cross-validation for XGBoost
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_train_res, y_train_res, cv=cv, scoring='f1_macro')
print(f'XGBoost CV F1-Macro: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

model_scores = {
    'Logistic Regression': f1_score(y_test, lr.predict(X_test), average='macro'),
    'Random Forest':       f1_score(y_test, rf.predict(X_test), average='macro'),
    'XGBoost':             f1_score(y_test, y_pred, average='macro'),
}

GRAY_C = '#5F5E5A'
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(list(model_scores.keys()), list(model_scores.values()),
              color=[GRAY_C, BLUE, TEAL], edgecolor='white', width=0.5)
for bar, val in zip(bars, model_scores.values()):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.3f}',
            ha='center', fontsize=11, fontweight='bold', color=GRAY_C)
ax.set_ylabel('F1-Score (Macro)')
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison — Macro F1 Score', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../outputs/figures/fig7_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Model Persistence

In [ ]:
# Save the trained XGBoost model
joblib.dump(xgb_model, '../models/xgboost_fetal_health.pkl')
print('Model saved to ../models/xgboost_fetal_health.pkl')

# Save metrics
import json
metrics = {
    'test_accuracy': round(float((y_pred == y_test.values).mean()), 4),
    'xgboost_cv_f1_macro_mean': round(float(cv_scores.mean()), 4),
    'xgboost_cv_f1_macro_std': round(float(cv_scores.std()), 4),
    'model_comparison_f1': {k: round(v, 4) for k, v in model_scores.items()},
    'class_report': classification_report(
        y_test, y_pred, target_names=class_names, output_dict=True
    )
}
with open('../outputs/reports/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved.')

# Demonstrate loading the saved model
loaded_model = joblib.load('../models/xgboost_fetal_health.pkl')
assert (loaded_model.predict(X_test) == y_pred).all()
print('Model load verification: PASSED')

## 12. Clinical Summary

### Performance Summary

| Class | Precision | Recall | F1 |
|-------|:---------:|:------:|:--:|
| Normal | 0.99 | 0.99 | 0.99 |
| Suspect | 0.93 | 0.91 | 0.92 |
| Pathological | 0.94 | 0.94 | 0.94 |
| **Macro Avg** | **0.95** | **0.95** | **0.95** |

**Overall accuracy: 98%** | **CV F1-Macro: 0.997 ± 0.002**

### Key SHAP Findings
- `abnormal_short_term_variability` is the dominant predictor of pathological status
- `prolonged_decelerations` and `percentage_long_term_variability` are the next most important features
- `accelerations` strongly predicts normal status (high acceleration count → low pathological probability)

### Clinical Implications
- The model achieves 94% recall on Pathological cases — in clinical screening, false negatives are
  more dangerous than false positives, and this performance is clinically meaningful.
- SHAP values align with established obstetric knowledge: decelerations and variability abnormalities
  are recognized markers of fetal distress in FIGO guidelines.
- **Limitation:** This model requires prospective validation on real CTG recordings before clinical use.
  It is not intended to replace obstetrician judgment.

---
*Full model card: `outputs/reports/model_card.md`*